### Loading necessary libraries

In [1]:
import pandas as pd
import pyspark
from pyspark.sql import SparkSession, functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import StopWordsRemover, Tokenizer, NGram
from pyspark.ml.feature import HashingTF, MinHashLSH, RegexTokenizer, SQLTransformer

#Create PySpark SparkSession
spark = SparkSession.builder.master('local[*]').appName('Approximate String Matching').getOrCreate()

22/11/01 16:20:56 WARN Utils: Your hostname, ENG401516 resolves to a loopback address: 127.0.1.1; using 10.255.151.158 instead (on interface wlp2s0)
22/11/01 16:20:56 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
22/11/01 16:20:59 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
22/11/01 16:21:01 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


### Loading the original synthetic data

In [2]:
df = pd.read_csv('synthetic_data.csv')

### Creating a new field 'FullName' by incorporating 'FirstName' and 'LastName'

In [3]:
df['FullName'] = df['FirstName']+ ' ' + df['LastName']

In [4]:
df[['FullName']].head()

,FullName
0,CHARMAINE RIVERA
1,DEREK ASHLEY
2,STACIE PIEPER
3,MICHAEL RIGGLEMAN
4,KIM BORGHOFF


### Creating a new dataframe for the original synthetic data that has only 'VoterID' and 'FullName'

In [5]:
df_1 = df[['VoterID', 'FullName']]

In [6]:
df_1.head()

,VoterID,FullName
0,656306991,CHARMAINE RIVERA
1,143659045,DEREK ASHLEY
2,11724731,STACIE PIEPER
3,385589500,MICHAEL RIGGLEMAN
4,400704251,KIM BORGHOFF


In [7]:
df_1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   VoterID   10000 non-null  int64 
 1   FullName  10000 non-null  object
dtypes: int64(1), object(1)
memory usage: 156.4+ KB


### Loading the synthetic data after applying misinformation

In [8]:
dff = pd.read_csv('synthetic_data_2.csv')

### Insight of the loaded data

In [9]:
dff.head()

,VoterID,BirthDate,Age,LAST4SSN,DriverLicCard,Prefix,FirstName,LastName,Suffix,MiddleName,...,MailResAddr2,MailResCity,MailResZip,MailresState,MailResCountry,RegistrationDate,PartyDesc,precinct,Split,PrecinctLabel
0,656306991,30/05/1963,59,5247,YP737187G,Mr.,CHARMAINE,RIVERA,Sr.,NaN,...,NaN,PARIS,83202,ID,NaN,29/05/1981,Libertarian,2263,49.0,2263
1,143659045,14/08/1982,40,3236,BJ641044J,Mr.,DEREK,ASHLEY,NaN,SANDRA,...,NaN,MCCALL,83541,ID,NaN,13/08/2000,Constitution,9469,48.0,9469
2,11724731,06/12/1987,34,6873,QG306251E,Mr.,STACIE,PIEPER,Jr.,NaN,...,NaN,BERN,83316,ID,United States,05/12/2005,Democratic,5955,16.0,5955
3,385589500,24/12/1962,59,8473,NaN,Mrs.,RIGGLEMAN,MICHAEL,NaN,NaN,...,NaN,NAPLES,83239,ID,NaN,23/12/1980,Libertarian,8726,26.0,8726
4,400704251,16/02/1939,83,8189,XP239943X,Mr.,K,BORGHOFF,Jr.,NaN,...,NaN,WORLEY,83232,ID,NaN,15/02/1957,Unaffiliated,9010,9.0,9010


In [10]:
dff.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 28 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   VoterID           10000 non-null  int64  
 1   BirthDate         10000 non-null  object 
 2   Age               10000 non-null  int64  
 3   LAST4SSN          10000 non-null  int64  
 4   DriverLicCard     5022 non-null   object 
 5   Prefix            7515 non-null   object 
 6   FirstName         10000 non-null  object 
 7   LastName          10000 non-null  object 
 8   Suffix            6716 non-null   object 
 9   MiddleName        3367 non-null   object 
 10  Gender            10000 non-null  object 
 11  PhoneNumber       4886 non-null   object 
 12  ResStreetAddress  10000 non-null  object 
 13  ResCityDesc       10000 non-null  object 
 14  ResState          10000 non-null  object 
 15  ResZip5           10000 non-null  int64  
 16  ResCountyDesc     10000 non-null  object 

### Creating a new field 'FullName' by incorporating 'FirstName' and 'LastName' for this dataset

In [11]:
dff['FullName'] = dff['FirstName']+ ' ' + dff['LastName']

In [12]:
dff[['FullName']].head()

,FullName
0,CHARMAINE RIVERA
1,DEREK ASHLEY
2,STACIE PIEPER
3,RIGGLEMAN MICHAEL
4,K BORGHOFF


### Creating a new dataframe for the second synthetic data that contains only 'VoterID' and 'FullName'

In [13]:
df_2 = dff[['VoterID', 'FullName']]

In [14]:
df_2.head()

,VoterID,FullName
0,656306991,CHARMAINE RIVERA
1,143659045,DEREK ASHLEY
2,11724731,STACIE PIEPER
3,385589500,RIGGLEMAN MICHAEL
4,400704251,K BORGHOFF


In [15]:
df_2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   VoterID   10000 non-null  int64 
 1   FullName  10000 non-null  object
dtypes: int64(1), object(1)
memory usage: 156.4+ KB


### Checking how many records between these two datasets are matched on 'FullName'

In [16]:
result_1 = df_1.merge(df_2, on = 'FullName')
result_1

,VoterID_x,FullName,VoterID_y
0,656306991,CHARMAINE RIVERA,656306991
1,143659045,DEREK ASHLEY,143659045
2,11724731,STACIE PIEPER,11724731
3,136622358,JOSE LUIS,136622358
4,965805178,JOSEPH BLIND,965805178
...,...,...,...
9451,613652641,CAROL HOLMES,613652641
9452,971340310,CANDACE BIBEE,971340310
9453,587819515,TIMOTHY BOYD,587819515
9454,830497460,MARIA WOLFE,830497460


**It can be observed that out of 10000 records, 9456 records have been matched but the rest of the 544 records are not matched because of the misinformation applied on the original synthetic data**

### Converting both of the newly created dataframes into pyspark dataframes

In [17]:
df_1 = spark.createDataFrame(df_1)

In [18]:
df_1.show(5)

+---------+-----------------+
|  VoterID|         FullName|
+---------+-----------------+
|656306991| CHARMAINE RIVERA|
|143659045|     DEREK ASHLEY|
| 11724731|    STACIE PIEPER|
|385589500|MICHAEL RIGGLEMAN|
|400704251|     KIM BORGHOFF|
+---------+-----------------+
only showing top 5 rows



In [19]:
df_2 = spark.createDataFrame(df_2)

In [20]:
df_2.show(5)

+---------+-----------------+
|  VoterID|         FullName|
+---------+-----------------+
|656306991| CHARMAINE RIVERA|
|143659045|     DEREK ASHLEY|
| 11724731|    STACIE PIEPER|
|385589500|RIGGLEMAN MICHAEL|
|400704251|       K BORGHOFF|
+---------+-----------------+
only showing top 5 rows



### Applying a machine learning pipeline that involves transformation, tokenization, removing stop words, Ngram, and Jaccard distance calculation. 

In [21]:
model = Pipeline(stages=[
    SQLTransformer(statement="SELECT *, lower(FullName) lower FROM __THIS__"),
    Tokenizer(inputCol="lower", outputCol="token"),
    StopWordsRemover(inputCol="token", outputCol="stop"),
    SQLTransformer(statement="SELECT *, concat_ws(' ', stop) concat FROM __THIS__"),
    RegexTokenizer(pattern="", inputCol="concat", outputCol="char", minTokenLength=1),
    NGram(n=2, inputCol="char", outputCol="ngram"),
    HashingTF(inputCol="ngram", outputCol="vector"),
    MinHashLSH(inputCol="vector", outputCol="lsh", numHashTables=3, seed=101012022)
]).fit(df_1)

#### Applying pipline on 1st dataframe

In [22]:
df_1 = model.transform(df_1)
df_1 = df_1.filter(F.size(F.col("ngram")) > 0)

#### Applying pipline on 2nd dataframe

In [23]:
df_2 = model.transform(df_2)
df_2 = df_2.filter(F.size(F.col("ngram")) > 0)

#### Joining them together by supplying a maximum Jaccard Distance of 0.5 that results in a match

In [24]:
result = model.stages[-1].approxSimilarityJoin(df_1, df_2, 0.6, "jaccardDist")
(result.select('datasetA.VoterID', 'datasetA.FullName', 'datasetB.VoterID', 'datasetB.FullName', 'jaccardDist').show(5))

/home/nahidanwar/spark-install/spark/python/lib/pyspark.zip/pyspark/sql/context.py:125: FutureWarning: Deprecated in 3.0.0. Use SparkSession.builder.getOrCreate() instead.


+---------+----------------+---------+----------------+------------------+
|  VoterID|        FullName|  VoterID|        FullName|       jaccardDist|
+---------+----------------+---------+----------------+------------------+
|470666898|      STEVEN PEW|788053618|   STEVEN WEAVER|0.5714285714285714|
|778666201| LATORIA GARLAND|778666201| LATORIA GARLAND|               0.0|
|611486116|  KATHY GONZALES|224844071|  BILLY GONZALEZ|0.5555555555555556|
|897986005|GREGORY GEARHART|897986005|GREGORY GEARHART|               0.0|
|737575052|   TWILA GILMORE|737575052|   TWILA GILMORE|               0.0|
+---------+----------------+---------+----------------+------------------+
only showing top 5 rows



#### Final matching with the minimum distance

In [25]:
from pyspark.sql import Window
w = Window.partitionBy('datasetA.VoterID')
result = (result.withColumn('minDist', F.min('jaccardDist').over(w))
           .where(F.col('jaccardDist') == F.col('minDist')).drop('minDist'))
(result.select('datasetA.FullName', 'datasetB.FullName', 'jaccardDist').show(5))

+--------------+--------------+-----------+
|      FullName|      FullName|jaccardDist|
+--------------+--------------+-----------+
|   MAYRA BATIE|   MAYRA BATIE|        0.0|
|MARGERY KEYSER|MARGERY KEYSER|        0.0|
|   DEVIN GEROW|   DEVIN GEROW|        0.0|
|KATIE WILLIAMS|KATIE WILLIAMS|        0.0|
| RICHARD PIRES| RICHARD PIRES|        0.0|
+--------------+--------------+-----------+
only showing top 5 rows



### Result

In [26]:
result.show()

+--------------------+--------------------+-------------------+
|            datasetA|            datasetB|        jaccardDist|
+--------------------+--------------------+-------------------+
|{1498017, ISABEL ...|{1498017, ISABEL ...|                0.0|
|{1772222, KELSEY ...|{1772222, KELSEY ...|                0.0|
|{2293540, ROSS ME...|{2293540, ROSS ME...|                0.0|
|{3425916, HARRIET...|{3425916, HARRIET...|                0.0|
|{3586842, ETHEL H...|{3586842, ETHEL H...|                0.0|
|{4338349, CHARLIE...|{4338349, CHARLIE...|                0.0|
|{4644780, RICHARD...|{4644780, RICHARD...|                0.0|
|{4685919, MELISSA...|{589589749, MELIS...|                0.0|
|{4685919, MELISSA...|{4685919, MELISSA...|                0.0|
|{5095122, RONALD ...|{5095122, RONALD ...|                0.0|
|{5364969, MIKE VI...|{5364969, MIKE VI...|                0.0|
|{5949488, SUSAN B...|{602158429, SUSAN...|                0.0|
|{5949488, SUSAN B...|{5949488, SUSAN B.

In [27]:
res = \
result.select('datasetA.VoterID', 'datasetA.FullName', 'datasetB.VoterID', 'datasetB.FullName', 'jaccardDist')

In [28]:
res.filter(F.col('jaccardDist') > 0).count()

825

### Approximate string matching results less than Jaccard distance of 0.5

In [29]:
res.filter(F.col('jaccardDist') > 0).show(20)

+--------+-----------------+---------+-----------------+-------------------+
| VoterID|         FullName|  VoterID|         FullName|        jaccardDist|
+--------+-----------------+---------+-----------------+-------------------+
| 6134056|     BRENDA AUTEN|  6134056|     AUTEN BRENDA|0.33333333333333337|
| 8029393|   BARBARA MILLER|301892593|     BARBARA HILL|0.46153846153846156|
| 8029393|   BARBARA MILLER|  8029393|       RBr MILLER|0.46153846153846156|
| 8148610|    HELEN WADDELL|  8148610|    WADDELL HELEN| 0.3076923076923077|
|26562160|    CRYSTAL GATES|898586856|     KRYSTAL GALE| 0.4285714285714286|
|32722294|    MAURICE HORNE|380959390|     MAURICE HALL| 0.4666666666666667|
|39209422|    WEI HEABERLIN| 39209422|    HEABERLIN WEI| 0.2857142857142857|
|39802452|      ANDRE SMITH|947286542|    MILDRED SMITH|                0.5|
|39802452|      ANDRE SMITH|778887506|        DAN SMITH|                0.5|
|39802452|      ANDRE SMITH|294508530|        JAN SMITH|                0.5|

In [30]:
res = res.filter(F.col('jaccardDist') > 0)

In [31]:
pandasDF = res.toPandas()
print(pandasDF)

       VoterID        FullName    VoterID       FullName  jaccardDist
0      6134056    BRENDA AUTEN    6134056   AUTEN BRENDA     0.333333
1      8029393  BARBARA MILLER  301892593   BARBARA HILL     0.461538
2      8029393  BARBARA MILLER    8029393     RBr MILLER     0.461538
3      8148610   HELEN WADDELL    8148610  WADDELL HELEN     0.307692
4     26562160   CRYSTAL GATES  898586856   KRYSTAL GALE     0.428571
..         ...             ...        ...            ...          ...
821  991388965        JUNE RAY  361158261     JUNE LAYNE     0.500000
822  992379552    ANITA HARRIS  482614785     ANN HARRIS     0.461538
823  994307736   LYNN CHAMBERS  994307736  CHAMBERS LYNN     0.285714
824  996921950      CAROL MEIR  613652641   CAROL HOLMES     0.538462
825  998633652   MABEL BONILLA  998633652   LMAr BONILLA     0.466667

[826 rows x 5 columns]


In [32]:
pandasDF.head(20)

,VoterID,FullName,VoterID,FullName,jaccardDist
0,6134056,BRENDA AUTEN,6134056,AUTEN BRENDA,0.333333
1,8029393,BARBARA MILLER,301892593,BARBARA HILL,0.461538
2,8029393,BARBARA MILLER,8029393,RBr MILLER,0.461538
3,8148610,HELEN WADDELL,8148610,WADDELL HELEN,0.307692
4,26562160,CRYSTAL GATES,898586856,KRYSTAL GALE,0.428571
5,32722294,MAURICE HORNE,380959390,MAURICE HALL,0.466667
6,39209422,WEI HEABERLIN,39209422,HEABERLIN WEI,0.285714
7,39802452,ANDRE SMITH,947286542,MILDRED SMITH,0.500000
8,39802452,ANDRE SMITH,778887506,DAN SMITH,0.500000
9,39802452,ANDRE SMITH,294508530,JAN SMITH,0.500000


In [33]:
pandasDF.columns = ['VoterID-A','FullName-A','VoterID-B','FullName-B','jaccardDist']

In [34]:
pandasDF.head(20)

,VoterID-A,FullName-A,VoterID-B,FullName-B,jaccardDist
0,6134056,BRENDA AUTEN,6134056,AUTEN BRENDA,0.333333
1,8029393,BARBARA MILLER,301892593,BARBARA HILL,0.461538
2,8029393,BARBARA MILLER,8029393,RBr MILLER,0.461538
3,8148610,HELEN WADDELL,8148610,WADDELL HELEN,0.307692
4,26562160,CRYSTAL GATES,898586856,KRYSTAL GALE,0.428571
5,32722294,MAURICE HORNE,380959390,MAURICE HALL,0.466667
6,39209422,WEI HEABERLIN,39209422,HEABERLIN WEI,0.285714
7,39802452,ANDRE SMITH,947286542,MILDRED SMITH,0.500000
8,39802452,ANDRE SMITH,778887506,DAN SMITH,0.500000
9,39802452,ANDRE SMITH,294508530,JAN SMITH,0.500000


In [35]:
pandasDF[pandasDF['VoterID-A'].eq(pandasDF['VoterID-B'])]

,VoterID-A,FullName-A,VoterID-B,FullName-B,jaccardDist
0,6134056,BRENDA AUTEN,6134056,AUTEN BRENDA,0.333333
2,8029393,BARBARA MILLER,8029393,RBr MILLER,0.461538
3,8148610,HELEN WADDELL,8148610,WADDELL HELEN,0.307692
6,39209422,WEI HEABERLIN,39209422,HEABERLIN WEI,0.285714
14,40436447,ROBERT RAMSAY,40436447,RAMSAY ROBERT,0.153846
...,...,...,...,...,...
817,980886919,JEFF RIDDICK,980886919,J RIDDICK,0.416667
819,985341558,ELIZABETH MCPEAK,985341558,MCPEAK ELIZABETH,0.235294
820,986659649,BOBBY PHILLIPS,986659649,PHILLIPS BOBBY,0.266667
823,994307736,LYNN CHAMBERS,994307736,CHAMBERS LYNN,0.285714


#### Out of 544 changed data 517 records have been identified using ASM